In [1]:
# Clear GPU memory (run this cell to start fresh)
import torch
import gc

# Clear Python garbage collector
gc.collect()

# Clear PyTorch CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared")
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
else:
    print("No CUDA device available")

GPU cache cleared
GPU memory allocated: 0.00 GB
GPU memory reserved: 0.00 GB


In [1]:
pip install -U bitsandbytes>=0.46.1


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import kagglehub
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download model files from Kaggle Hub
base_path = kagglehub.model_download("kevintang2048/blip2-explorer/pyTorch/default")
model_root = os.path.join(base_path, "blip2_lora_uicd")
processor_path = os.path.join(model_root, "processor")
model_path = os.path.join(model_root, "merged_model")

print("Model path:", model_path)
print("Files in model path:", os.listdir(model_path))

# Kaggle cache directory
cache_dir = "/kaggle/working/hf_cache"
os.makedirs(cache_dir, exist_ok=True)
print(f"Using cache directory: {cache_dir}")

# 8-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16
)

# Load model
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map={"": 0},
    cache_dir=cache_dir
)

# Load processor
processor = AutoProcessor.from_pretrained(
    processor_path,
    cache_dir=cache_dir
)

Using device: cuda
Model path: /kaggle/input/models/kevintang2048/blip2-explorer/pytorch/default/2/blip2_lora_uicd/merged_model
Files in model path: ['config.json', 'model.safetensors', 'generation_config.json']
Using cache directory: /kaggle/working/hf_cache


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

In [3]:
import zipfile
import os
import shutil
from pathlib import Path

# --- Colab-temp paths (fast, does not use Drive quota) ---

extract_root = Path("/kaggle/input/datasets/masonliang/uic-zip")


captions_matches = list(extract_root.rglob("UIC-captions.txt"))
if not captions_matches:
    raise FileNotFoundError("UIC-captions.txt not found after extraction (check zip contents).")

captions_file = captions_matches[0]
dataset_base = captions_file.parent
image_dir = dataset_base / "uic_224x224_image"

if not image_dir.exists():
    raise FileNotFoundError(f"Image directory not found: {image_dir}")

print("Dataset base:", dataset_base)
print("Captions file:", captions_file)
print("Image dir:", image_dir)

# --- Load captions ---
def load_captions(captions_path: Path):
    image_captions = {}
    with open(captions_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            img_id, caption = line.split(" ", 1)
            img_filename = img_id.split("#")[0]
            image_captions.setdefault(img_filename, []).append(caption)
    return image_captions

captions_dict = load_captions(captions_file)

# --- Build dataset list ---
image_paths = sorted([p.name for p in image_dir.iterdir() if p.suffix.lower() == ".jpg"])

dataset = []
for img_filename in image_paths:
    caps = captions_dict.get(img_filename, [])
    if caps:
        dataset.append({
            "image_path": str(image_dir / img_filename),
            "image_filename": img_filename,
            "captions": caps
        })

print(f"Loaded {len(dataset)} images with captions")
if dataset:
    print("Example entry:")
    print(f"  Image: {dataset[0]['image_filename']}")
    print(f"  Number of captions: {len(dataset[0]['captions'])}")
    print(f"  First caption: {dataset[0]['captions'][0]}")


Dataset base: /kaggle/input/datasets/masonliang/uic-zip/UIC(underwater image captioning dataset)
Captions file: /kaggle/input/datasets/masonliang/uic-zip/UIC(underwater image captioning dataset)/UIC-captions.txt
Image dir: /kaggle/input/datasets/masonliang/uic-zip/UIC(underwater image captioning dataset)/uic_224x224_image
Loaded 3176 images with captions
Example entry:
  Image: uic_img_1.jpg
  Number of captions: 5
  First caption: A dark brown turtle paddles through the water with its limbs .


In [4]:
import random

SEED = 42

# 1) Stable order first (so results don't depend on filesystem order)
dataset_sorted = sorted(dataset, key=lambda x: x["image_filename"])

# 2) Deterministic shuffle
rng = random.Random(SEED)
rng.shuffle(dataset_sorted)

# 3) 80/10/10 split
n = len(dataset_sorted)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val  # remainder

train_set = dataset_sorted[:n_train]
val_set   = dataset_sorted[n_train:n_train + n_val]
test_set  = dataset_sorted[n_train + n_val:]

print("Counts:", len(train_set), len(val_set), len(test_set))
print("First test example:", test_set[0]["image_filename"])


Counts: 2540 317 319
First test example: uic_img_205.jpg


In [5]:
import torch
from PIL import Image
from tqdm import tqdm

torch.manual_seed(42)

predictions = []

for item in tqdm(test_set):
    image = Image.open(item["image_path"]).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=3,        # keep fixed
            do_sample=False     # deterministic
        )

    caption = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    predictions.append({
        "image_filename": item["image_filename"],
        "prediction": caption,
        "references": item["captions"]
        "image_path": p["image_path"],
    })

print("Finished inference on test set")
print("Example output:")
print(predictions[0])


  0%|          | 0/319 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 319/319 [47:53<00:00,  9.01s/it]

Finished inference on test set
Example output:
{'image_filename': 'uic_img_205.jpg', 'prediction': 'A group of small fish swim around the coral . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .', 'references': ['Many brightly colored tropical fish swim among the corals .', 'A school of small fish swims in the water near the coral on the seabed .', 'There is a large area of coral on the seabed, and many small fish swim near it .', 'Groups of corals swim around near the corals on the seafloor .', 'Many small fish swim near the coral on the seabed .']}


In [8]:
# Make sure each prediction has image_path (needed for CLIPScore)
# If you already stored it, skip this block.
path_by_name = {x["image_filename"]: x["image_path"] for x in test_set}
for p in predictions:
    p["prediction"] = p["prediction"].strip()
    p["image_path"] = path_by_name[p["image_filename"]]

# For text metrics
pred_texts = [p["prediction"] for p in predictions]
ref_texts  = [p["references"] for p in predictions]  # list-of-lists

In [10]:
import json, time

output_path = "/kaggle/working/uic_blip2_augmented_predictions.json"

out = {
    "created_utc": time.time(),
    "seed": 42,
    "split": {
        "test_filenames": [p["image_filename"] for p in predictions],
    },
    "predictions": [
        {
            "image_filename": p["image_filename"],
            "prediction": p["prediction"].strip(),
            "references": [r.strip() for r in p["references"]],
        }
        for p in predictions
    ],
}

with open(output_path, "w") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"Saved to {output_path}")

Saved to /kaggle/working/uic_blip2_augmented_predictions.json


In [28]:
## This block is only for restoring if the session crashes after predictios are already saved from previous block
import zipfile
import os
import shutil
import json
from pathlib import Path


import json
import re

input_file = "/kaggle/working/uic_blip2_augmented_predictions.json"
output_file = "/kaggle/working/uic_blip2_augmented_predictions_clean.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

def clean_caption(text):
    text = text.strip()

    # collapse repeated ". . . ." patterns
    text = re.sub(r"(?:\s*\.\s*)+$", " .", text)

    return text

for p in data["predictions"]:
    p["prediction"] = clean_caption(p["prediction"])

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Cleaned JSON saved to:", output_file)

if not image_dir.exists():
    raise FileNotFoundError(f"Image directory not found: {image_dir}")

image_path_map = {
    p.name: str(p)
    for p in image_dir.iterdir()
    if p.suffix.lower() == ".jpg"
}

# Reload saved predictions
with open(predictions_json_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

predictions = []
missing = []

for p in saved["predictions"]:
    filename = p["image_filename"]
    img_path = image_path_map.get(filename)

    if img_path is None:
        missing.append(filename)
        continue

    predictions.append({
        "image_filename": filename,
        "image_path": p["image_path"],
        "prediction": p["prediction"].strip(),
        "references": [r.strip() for r in p["references"]],
    })

print("Recovered predictions:", len(predictions))
print("Missing images:", len(missing))
if predictions:
    print("Example recovered entry:", predictions[0])

Cleaned JSON saved to: /kaggle/working/uic_blip2_augmented_predictions_clean.json


FileNotFoundError: [Errno 2] No such file or directory: '/content/uic_blip2_baseline_predictions_clean.json'

In [17]:
#Otherwise use this block to clean the predictions
import json
import re

input_file = "/kaggle/working/uic_blip2_augmented_predictions.json"
output_file = "/kaggle/working/uic_blip2_augmented_predictions_clean.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

def clean_caption(text):
    text = text.strip()
    # collapse repeated ". . . ." patterns
    text = re.sub(r"(?:\s*\.\s*)+$", " .", text)
    return text

for p in data["predictions"]:
    p["prediction"] = clean_caption(p["prediction"])

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Cleaned JSON saved to:", output_file)

Cleaned JSON saved to: /kaggle/working/uic_blip2_augmented_predictions_clean.json


In [32]:
# data should already be loaded from your cleaned JSON file

image_dir = "/kaggle/input/datasets/masonliang/uic-zip/UIC(underwater image captioning dataset)/uic_224x224_image"

eval_predictions = []
for p in data["predictions"]:
    eval_predictions.append({
        "image_filename": p["image_filename"],
        "image_path": os.path.join(image_dir, p["image_filename"]),
        "prediction": p["prediction"],
        "references": p["references"],
    })

print(eval_predictions[0]["prediction"])
output_path = "/kaggle/working/uic_predictions_with_paths.json"

out = {
    "predictions": eval_predictions
}

with open(output_path, "w") as f:
    json.dump(out, f, indent=2)

print("Saved:", output_path)
predictions = eval_predictions
scores = []

with torch.no_grad():
    for p in tqdm(eval_predictions):
        img = preprocess(Image.open(p["image_path"]).convert("RGB")).unsqueeze(0).to(device)
        txt = tokenizer([p["prediction"]]).to(device)

        img_feat = model_clip.encode_image(img)
        txt_feat = model_clip.encode_text(txt)

        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

        sim = (img_feat @ txt_feat.T).item()
        scores.append(sim)

clipscore_mean = sum(scores) / len(scores)
print("CLIPScore mean:", clipscore_mean)



A group of small fish swim around the coral .
Saved: /kaggle/working/uic_predictions_with_paths.json


100%|██████████| 319/319 [00:05<00:00, 54.05it/s]

CLIPScore mean: 0.31263058779755354


In [33]:
!pip install pycocoevalcap
!pip install pycocotools


# Build dictionaries indexed by image id
gts = {}
res = {}

for idx, p in enumerate(eval_predictions):
    gts[idx] = [ref.strip() for ref in p["references"]]
    res[idx] = [p["prediction"].strip()]

In [34]:
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
!apt-get update -qq
!apt-get install -y openjdk-11-jre-headless

!update-alternatives --set java /usr/lib/jvm/java-11-openjdk-amd64/bin/java


cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(gts, res)
print("CIDEr:", cider_score)

spice_scorer = Spice()
spice_score, _ = spice_scorer.compute_score(gts, res)
print("SPICE:", spice_score)

meteor_scorer = Meteor()
meteor_score, _ = meteor_scorer.compute_score(gts, res)
print("METEOR:", meteor_score)

bleu_scorer = Bleu(4)
bleu_scores, _ = bleu_scorer.compute_score(gts, res)
print("BLEU-1..4:", bleu_scores)

rouge_scorer = Rouge()
rouge_score, _ = rouge_scorer.compute_score(gts, res)
print("ROUGE-L:", rouge_score)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jre-headless is already the newest version (11.0.30+7-1ubuntu1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 203 not upgraded.
CIDEr: 1.3173646423945018


Parsing reference captions
Parsing test captions
Initiating Stanford parsing pipeline
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator tokenize
[main] INFO edu.stanford.nlp.pipeline.TokenizerAnnotator - TokenizerAnnotator: No tokenizer type provided. Defaulting to PTBTokenizer.
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ssplit
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator parse
[main] INFO edu.stanford.nlp.parser.common.ParserGrammar - Loading parser from serialized file edu/stanford/nlp/models/lexparser/englishPCFG.ser.gz ... 
done [0.7 sec].
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator lemma
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ner
Loading classifier from edu/stanford/nlp/models/ner/english.all.3class.distsim.crf.ser.gz ... done [1.5 sec].
Loading classifier from edu/stanford/nlp/models/ner/english.muc.7class.distsim.crf.ser.gz ... done [0.8

SPICE evaluation took: 19.66 s
SPICE: 0.2668355028586813
METEOR: 0.333275909391198
{'testlen': 3517, 'reflen': 3500, 'guess': [3517, 3198, 2879, 2560], 'correct': [2743, 1706, 1054, 623]}
ratio: 1.0048571428568558
BLEU-1..4: [0.7799260733577538, 0.6450256771527308, 0.5340529490231333, 0.43878347019397235]
ROUGE-L: 0.663384021364908


In [35]:
print(f"CLIP similarity (mean): {clipscore_mean:.4f}")
print(f"CIDEr: {cider_score:.4f}")
print(f"SPICE: {spice_score:.4f}")
print(f"METEOR: {meteor_score:.4f}")
print(f"BLEU: {bleu_scores[0]:.4f} {bleu_scores[1]:.4f} {bleu_scores[2]:.4f} {bleu_scores[3]:.4f}")
print(f"ROUGE: {rouge_score:.4f}")


CLIP similarity (mean): 0.3126
CIDEr: 1.3174
SPICE: 0.2668
METEOR: 0.3333
BLEU: 0.7799 0.6450 0.5341 0.4388
ROUGE: 0.6634
